# 🎨 DiffuSVG: Experimental Pipeline v4.1

⚠️ **BEFORE RUNNING**:
1. `Runtime > Change runtime type > T4 GPU` (A100 recommended)
2. **Get HuggingFace Token** (for SD 3.5 Medium):
   - Visit https://huggingface.co/settings/tokens
   - Create a token (read access)
   - Accept license at https://huggingface.co/stabilityai/stable-diffusion-3.5-medium
3. Enter token in Configuration cell below

**Models Used:**
- 🖼️ **Diffusion**: Stable Diffusion 3.5 Medium (latest, best quality)
- 🤖 **VLM**: Qwen2-VL-7B-Instruct (runs locally, ~15GB)

In [ ]:
#@title 🔑 **Configuration** { display-mode: "form" }

# HuggingFace token for SD 3.5 Medium (get from https://huggingface.co/settings/tokens)
HF_TOKEN = ""  #@param {type:"string"}

NUM_TEST_PROMPTS = 50  #@param {type:"slider", min:10, max:200, step:10}
MAX_REFINEMENT_ITERATIONS = 2  #@param {type:"slider", min:0, max:5, step:1}

# Diffusion settings (optimized for SVG-friendly output)
DIFFUSION_STEPS = 30  #@param {type:"slider", min:20, max:50, step:5}
GUIDANCE_SCALE = 5.0  #@param {type:"slider", min:3, max:10, step:0.5}
SVG_SIMPLIFICATION = "high"  #@param ["low", "medium", "high"]

# VLM settings
QWEN_PRECISION = "4bit"  #@param ["4bit", "8bit", "fp16"]
MAX_SVG_TOKENS = 4096  #@param {type:"slider", min:2048, max:8192, step:1024}

# Validate HF token
if not HF_TOKEN or len(HF_TOKEN) < 20:
    print("⚠️ HuggingFace token not set!")
    print("   1. Visit: https://huggingface.co/settings/tokens")
    print("   2. Create token (read access)")
    print("   3. Accept license: https://huggingface.co/stabilityai/stable-diffusion-3.5-medium")
    print("   4. Paste token above")
else:
    # Login to HuggingFace
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print(f"✅ HuggingFace authenticated (token: {HF_TOKEN[:10]}...)")

print(f"✅ Config loaded:")
print(f"   📊 Test prompts: {NUM_TEST_PROMPTS}")
print(f"   🎨 Diffusion steps: {DIFFUSION_STEPS}, guidance: {GUIDANCE_SCALE}")
print(f"   🤖 Qwen precision: {QWEN_PRECISION}, max tokens: {MAX_SVG_TOKENS}")
print(f"   🔧 SVG simplification: {SVG_SIMPLIFICATION}")

In [ ]:
#@title 📦 **Install Dependencies** (Run ONCE, then RESTART)
# Uninstall old versions first
!pip uninstall -y transformers accelerate bitsandbytes diffusers huggingface_hub -q 2>/dev/null

# Install compatible versions
!pip install huggingface_hub>=0.26.2 -U -q
!pip install transformers==4.45.0 accelerate==0.34.0 -q
!pip install bitsandbytes -q
!pip install diffusers==0.31.0 -q
!pip install qwen-vl-utils pillow torchvision -q
!pip install optimum triton -q
!pip install cairosvg numpy pandas matplotlib tqdm -q
!pip install open_clip_torch -q
!pip install peft -q
!apt-get install -qq libcairo2-dev > /dev/null 2>&1

print("\n" + "="*60)
print("✅ DEPENDENCIES INSTALLED!")
print("="*60)
print("🔴 MANDATORY NEXT STEP:")
print("   1. Runtime > Restart runtime")
print("   2. After restart, run Cell 2 (Config)")
print("   3. Then run Cell 4 → 5 → 6 → 7 → 8")
print("="*60)
print("\n⚠️  DO NOT skip the restart or you'll get errors!")

In [ ]:
#@title 📚 **Imports & GPU Check**
import os, io, re, json, base64, random, warnings, time, traceback
from typing import Optional, Dict, Any
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cairosvg
import torch
import torch.nn.functional as F
import open_clip
warnings.filterwarnings('ignore')

# GPU check
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB)")
else:
    raise RuntimeError("❌ No GPU! Runtime > Change runtime type > T4 GPU")

print("✅ Imports done")

In [ ]:
#@title 🔌 **Load Qwen2-VL** (~5 min first time, then cached)

print("Loading Qwen2-VL-7B-Instruct...")

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch

# Quantization config for memory efficiency
if QWEN_PRECISION == "4bit":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4"
    )
elif QWEN_PRECISION == "8bit":
    quant_config = BitsAndBytesConfig(load_in_8bit=True)
else:
    quant_config = None

model_name = "Qwen/Qwen2-VL-7B-Instruct"
print(f"   Model: {model_name}")
print(f"   Precision: {QWEN_PRECISION}")
print(f"   Size: ~15GB (compressed to ~4GB with 4bit)")
print(f"   Downloading (may take 5-10 min)...")

try:
    # Load model directly (handles download automatically)
    print(f"   Loading model...")
    qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.float16 if quant_config is None else None
    )
    
    print(f"   Loading processor...")
    qwen_processor = AutoProcessor.from_pretrained(model_name)
    
    print(f"\n✅ Qwen2-VL loaded ({QWEN_PRECISION})")
    print(f"   GPU Memory: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    
except Exception as e:
    print(f"\n❌ Failed to load Qwen2-VL: {e}")
    print(f"\n💡 TROUBLESHOOTING:")
    print(f"   1. Check GPU memory: Runtime > View resources")
    print(f"   2. Try 4bit precision (lowest memory)")
    print(f"   3. Restart runtime: Runtime > Restart runtime")
    print(f"   4. Run cells in order: 2 → 3 → 4 → 5")
    raise

In [ ]:
#@title 🛠️ **SVG Utilities**

class SVGUtils:
    @staticmethod
    def extract_svg(text: str) -> Optional[str]:
        if not text:
            return None
        # Try multiple patterns
        patterns = [
            r'```(?:svg|xml)?\n?([\s\S]*?)```',  # Code block
            r'<svg[^>]*>[\s\S]*?</svg>',  # Direct SVG
        ]
        for p in patterns:
            m = re.search(p, text, re.IGNORECASE)
            if m:
                content = m.group(1) if '```' in p else m.group(0)
                if '<svg' in content.lower():
                    # Extract just the SVG part
                    svg_match = re.search(r'<svg[^>]*>[\s\S]*?</svg>', content, re.IGNORECASE)
                    if svg_match:
                        return svg_match.group(0).strip()
        # Last resort: check if whole text is SVG
        if text.strip().lower().startswith('<svg'):
            return text.strip()
        return None
    
    @staticmethod
    def validate(svg: str) -> bool:
        if not svg or '<svg' not in svg.lower():
            return False
        try:
            cairosvg.svg2png(bytestring=svg.encode(), output_width=64, output_height=64)
            return True
        except:
            return False
    
    @staticmethod
    def repair(svg: str) -> str:
        if not svg:
            return '<svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg"></svg>'
        if 'xmlns' not in svg:
            svg = svg.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
        if 'viewbox' not in svg.lower():
            svg = svg.replace('<svg', '<svg viewBox="0 0 200 200"', 1)
        return svg
    
    @staticmethod
    def render(svg: str, size: int = 512) -> Optional[Image.Image]:
        try:
            png = cairosvg.svg2png(bytestring=svg.encode(), output_width=size, output_height=size)
            return Image.open(io.BytesIO(png)).convert('RGB')
        except:
            return None
    
    @staticmethod
    def count_elements(svg: str) -> int:
        if not svg:
            return 0
        return len(re.findall(r'<(path|rect|circle|ellipse|polygon|line)[\s/>]', svg, re.I))

print("✅ SVGUtils ready")

In [ ]:
#@title 📊 **Load Metrics (CLIP + DINO)**

class Metrics:
    def __init__(self):
        print("Loading CLIP...")
        self.clip, _, self.clip_prep = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
        self.clip = self.clip.cuda().eval()
        self.tokenizer = open_clip.get_tokenizer('ViT-B-32')
        print("Loading DINO...")
        self.dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', verbose=False).cuda().eval()
        print("✅ Metrics ready")
    
    @torch.no_grad()
    def clip_score(self, img: Image.Image, text: str) -> float:
        img_t = self.clip_prep(img).unsqueeze(0).cuda()
        txt_t = self.tokenizer([text]).cuda()
        img_f = F.normalize(self.clip.encode_image(img_t), dim=-1)
        txt_f = F.normalize(self.clip.encode_text(txt_t), dim=-1)
        return (img_f @ txt_f.T).item() * 100
    
    @torch.no_grad()
    def dino_score(self, img1: Image.Image, img2: Image.Image) -> float:
        def prep(img):
            img = img.resize((224, 224))
            arr = (np.array(img)/255.0 - [0.485,0.456,0.406]) / [0.229,0.224,0.225]
            return torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).float().cuda()
        return max(0, F.cosine_similarity(self.dino(prep(img1)), self.dino(prep(img2))).item())

metrics = Metrics()

In [ ]:
#@title 🎨 **Load SD 3.5 Medium** (optimized for SVG)

print("Loading Stable Diffusion 3.5 Medium (~3 min)...")

from diffusers import StableDiffusion3Pipeline
import torch

pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3.5-medium",
    torch_dtype=torch.float16,
    token=HF_TOKEN  # Use authenticated token
)

# Enable CPU offloading to save VRAM (~10GB)
pipe.enable_model_cpu_offload()

# SVG-optimized prompt engineering
SVG_STYLE_PRESETS = {
    "low": "simple vector art, ",
    "medium": "flat vector illustration, minimal design, clean shapes, ",
    "high": "ultra-simple flat vector, geometric shapes only, solid colors, icon style, minimalist, "
}

STYLE = SVG_STYLE_PRESETS[SVG_SIMPLIFICATION]
NEG = "gradient, shadow, texture, 3d, realistic, photograph, complex details, noise, grain"

@torch.inference_mode()
def generate_image(prompt: str, seed: int = None) -> Image.Image:
    """Generate SVG-friendly image with SD 3.5 Medium"""
    gen = torch.Generator("cuda").manual_seed(seed) if seed else None
    
    # Optimized for flat, vector-like output
    return pipe(
        STYLE + prompt,
        negative_prompt=NEG,
        num_inference_steps=DIFFUSION_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=gen
    ).images[0]

print(f"✅ SD 3.5 Medium ready (CPU offloaded)")
print(f"   Style: {SVG_SIMPLIFICATION} ({STYLE.strip()})")
print(f"   Steps: {DIFFUSION_STEPS}, Guidance: {GUIDANCE_SCALE}")
print(f"   GPU Memory: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")

In [ ]:
#@title 🤖 **Qwen2-VL SVG Generator**

SYSTEM_PROMPT = """You are an expert SVG code generator. Convert the image to clean, minimal SVG code.

RULES:
- Output ONLY raw SVG code (no markdown, no explanation, no code blocks)
- Start with: <svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg">
- Use simple shapes: <rect>, <circle>, <ellipse>, <polygon>, <path>
- Use solid hex colors only (e.g., fill="#FF0000")
- Keep it minimal: 5-15 elements maximum
- End with: </svg>

Output the SVG code directly:"""

def vlm_to_svg(image: Image.Image, prompt: str) -> tuple:
    """Convert image to SVG using Qwen2-VL. Returns (svg_code, raw_response, error)"""
    
    try:
        # Prepare messages for Qwen2-VL
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": f"{SYSTEM_PROMPT}\n\nConvert this image to SVG. It shows: {prompt}"}
            ]
        }]
        
        # Apply chat template
        text = qwen_processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        # Prepare inputs
        inputs = qwen_processor(
            text=[text],
            images=[image],
            padding=True,
            return_tensors="pt"
        ).to("cuda")
        
        # Generate SVG code
        with torch.inference_mode():
            output_ids = qwen_model.generate(
                **inputs,
                max_new_tokens=MAX_SVG_TOKENS,
                do_sample=False,
                temperature=0.7,
                top_p=0.9
            )
        
        # Decode output
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(inputs.input_ids, output_ids)
        ]
        raw = qwen_processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0]
        
        # Extract and repair SVG
        svg = SVGUtils.extract_svg(raw)
        if svg:
            svg = SVGUtils.repair(svg)
        
        return svg, raw, None
        
    except Exception as e:
        return None, "", f"{type(e).__name__}: {e}"

print("✅ Qwen2-VL SVG generator ready")

In [ ]:
#@title 🧪 **Test Full Pipeline** (Run this!)

print("="*60)
print("Testing SD 3.5 Medium + Qwen2-VL pipeline")
print("="*60)

# Step 1: Generate image
print("\n1️⃣ Generating image with SD 3.5 Medium...")
# Move Qwen to CPU to free VRAM for SD 3.5
qwen_model.to('cpu')
torch.cuda.empty_cache()

test_img = generate_image("a red circle", seed=42)
print(f"   ✅ Image generated: {test_img.size}")
display(test_img.resize((256, 256)))

# Step 2: VLM to SVG
print("\n2️⃣ Calling Qwen2-VL...")
# Move Qwen back to GPU
qwen_model.to('cuda')
torch.cuda.empty_cache()

svg, raw, err = vlm_to_svg(test_img, "a red circle")

if err:
    print(f"   ❌ Error: {err}")
else:
    print(f"   ✅ Got response: {len(raw)} chars")
    print(f"   Raw response preview:")
    print(f"   {raw[:300]}{'...' if len(raw) > 300 else ''}")

# Step 3: Extract SVG
print("\n3️⃣ Extracting SVG...")
if svg:
    print(f"   ✅ Extracted: {len(svg)} chars, {SVGUtils.count_elements(svg)} elements")
    print(f"   {svg[:200]}...")
else:
    print(f"   ❌ Could not extract SVG")
    if raw:
        print(f"   Full response for debugging:")
        print(raw)

# Step 4: Validate & Render
print("\n4️⃣ Validating & Rendering...")
if svg:
    valid = SVGUtils.validate(svg)
    print(f"   Valid: {valid}")
    
    if valid:
        rendered = SVGUtils.render(svg)
        clip = metrics.clip_score(rendered, "a red circle")
        dino = metrics.dino_score(test_img, rendered)
        
        print(f"   CLIP: {clip:.2f}")
        print(f"   DINO: {dino:.3f}")
        
        # Show images
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))
        ax[0].imshow(test_img); ax[0].set_title('SD 3.5 Medium'); ax[0].axis('off')
        ax[1].imshow(rendered); ax[1].set_title(f'SVG Render (CLIP:{clip:.1f})'); ax[1].axis('off')
        # Show SVG code
        ax[2].text(0.05, 0.95, svg[:400] + "..." if len(svg) > 400 else svg, 
                   fontfamily='monospace', fontsize=6, verticalalignment='top', wrap=True)
        ax[2].set_title('SVG Code'); ax[2].axis('off')
        plt.tight_layout()
        plt.show()
        
        print("\n✅ Pipeline working! Proceed to full evaluation.")
    else:
        print(f"   ❌ SVG invalid. Content:")
        print(svg)
else:
    print("\n❌ Pipeline failed. Check errors above.")

In [ ]:
#@title 📝 **Test Prompts**
PROMPTS = [
    "a red apple", "a yellow sun", "a blue circle", "a green tree", "a red heart",
    "a yellow star", "an orange carrot", "a pink flower", "a house with red roof",
    "a snowman", "a rocket", "a cat face", "a wifi symbol", "a battery icon",
    "a music note", "a play button", "a gear icon", "a home icon", "a mail envelope",
    "a phone icon", "a camera", "a lock", "a mountain", "a rainbow", "clouds",
    "a crescent moon", "a pizza slice", "a coffee cup", "an ice cream", "a cake",
    "a hamburger", "a donut", "a watermelon", "a banana", "a strawberry",
    "a hot air balloon", "a treasure chest", "a lighthouse", "a bicycle", "a guitar",
    "circles", "a spiral", "squares", "yin yang", "a peace sign",
    "a target", "a smiley", "thumbs up", "lightning bolt", "a car"
]
random.seed(42)
selected = random.sample(PROMPTS, min(NUM_TEST_PROMPTS, len(PROMPTS)))
print(f"✅ {len(selected)} prompts selected")

In [ ]:
#@title 🚀 **Run Full Evaluation**

# Fix memory fragmentation issue
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

results = []
print(f"Running on {len(selected)} prompts...\n")
print("💡 Moving models between CPU/GPU to manage memory...\n")

import gc

for i, prompt in enumerate(tqdm(selected)):
    r = {'prompt': prompt, 'success': False, 'clip': 0, 'dino': 0, 'elements': 0,
         'time': 0, 'svg': '', 'error': None}
    
    try:
        t0 = time.time()
        
        # Move Qwen to CPU, aggressive memory clearing
        qwen_model.to('cpu')
        torch.cuda.empty_cache()
        gc.collect()
        
        # Generate image with SD 3.5
        img = generate_image(prompt, seed=42+i)
        
        # Clear SD memory before moving Qwen back
        torch.cuda.empty_cache()
        gc.collect()
        
        # Move Qwen back to GPU
        qwen_model.to('cuda')
        torch.cuda.empty_cache()
        
        # Get SVG
        svg, raw, err = vlm_to_svg(img, prompt)
        r['time'] = time.time() - t0
        
        if err:
            r['error'] = err
        elif svg and SVGUtils.validate(svg):
            rendered = SVGUtils.render(svg)
            if rendered:
                r['success'] = True
                r['svg'] = svg
                r['clip'] = metrics.clip_score(rendered, prompt)
                r['dino'] = metrics.dino_score(img, rendered)
                r['elements'] = SVGUtils.count_elements(svg)
                r['_rendered'] = rendered
            else:
                r['error'] = "SVG render failed"
        elif svg:
            r['error'] = "SVG validation failed"
        else:
            r['error'] = "SVG extraction failed"
        
        # Store image for later
        r['_img'] = img
        
        # Clean up to prevent memory leaks
        del img
        if 'rendered' in locals():
            del rendered
        torch.cuda.empty_cache()
        
    except Exception as e:
        r['error'] = str(e)
    
    results.append(r)
    
    # Progress
    if (i+1) % 10 == 0:
        valid = sum(1 for x in results if x['success'])
        avg_clip = np.mean([x['clip'] for x in results if x['success']]) if valid else 0
        print(f"  [{i+1}/{len(selected)}] Valid: {valid} ({100*valid/(i+1):.0f}%) | CLIP: {avg_clip:.1f}")

print("\n✅ Done!")

In [ ]:
#@title 📈 **Results Summary**

valid = [r for r in results if r['success']]
failed = [r for r in results if not r['success']]

print(f"\nTotal: {len(results)} | Valid: {len(valid)} | Failed: {len(failed)}\n")

if valid:
    summary = {
        'N': len(results),
        'Valid': len(valid),
        'Validity %': 100 * len(valid) / len(results),
        'CLIP Mean': np.mean([r['clip'] for r in valid]),
        'CLIP Std': np.std([r['clip'] for r in valid]),
        'DINO Mean': np.mean([r['dino'] for r in valid]),
        'DINO Std': np.std([r['dino'] for r in valid]),
        'Avg Elements': np.mean([r['elements'] for r in valid]),
        'Avg Time': np.mean([r['time'] for r in results]),
    }
    
    print("=" * 45)
    print("        DIFFUSVG RESULTS")
    print("=" * 45)
    for k, v in summary.items():
        print(f"  {k:.<25} {v:.2f}" if isinstance(v, float) else f"  {k:.<25} {v}")
    print("=" * 45)
else:
    print("❌ No valid results!")
    print("\nFirst 3 errors:")
    for r in failed[:3]:
        print(f"  {r['prompt']}: {r.get('error', 'Unknown')}")
    summary = {}

In [ ]:
#@title 📊 **Plots**
if len(valid) >= 3:
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    
    clips = [r['clip'] for r in valid]
    ax[0].hist(clips, bins=min(15, len(clips)), color='steelblue', edgecolor='black')
    ax[0].axvline(np.mean(clips), color='red', ls='--', label=f'μ={np.mean(clips):.1f}')
    ax[0].set_xlabel('CLIP Score'); ax[0].set_title('CLIP Distribution'); ax[0].legend()
    
    dinos = [r['dino'] for r in valid]
    ax[1].hist(dinos, bins=min(15, len(dinos)), color='seagreen', edgecolor='black')
    ax[1].axvline(np.mean(dinos), color='red', ls='--', label=f'μ={np.mean(dinos):.2f}')
    ax[1].set_xlabel('DINO Score'); ax[1].set_title('DINO Distribution'); ax[1].legend()
    
    ax[2].scatter(clips, dinos, alpha=0.6, c='purple')
    ax[2].set_xlabel('CLIP'); ax[2].set_ylabel('DINO'); ax[2].set_title('CLIP vs DINO')
    
    plt.tight_layout()
    plt.savefig('plots.png', dpi=150)
    plt.show()
else:
    print(f"Need ≥3 valid results (have {len(valid)})")

In [ ]:
#@title 🖼️ **Gallery**
if len(valid) >= 4:
    top = sorted(valid, key=lambda x: x['clip'], reverse=True)[:min(5, len(valid))]
    
    fig, ax = plt.subplots(2, len(top), figsize=(4*len(top), 8))
    for i, r in enumerate(top):
        ax[0,i].imshow(r['_img']); ax[0,i].set_title(r['prompt'][:18], fontsize=9); ax[0,i].axis('off')
        if r['_rendered']:
            ax[1,i].imshow(r['_rendered'])
        ax[1,i].set_title(f"CLIP:{r['clip']:.1f}", fontsize=9); ax[1,i].axis('off')
    
    plt.suptitle('Top Results', fontsize=14)
    plt.tight_layout()
    plt.savefig('gallery.png', dpi=150)
    plt.show()
else:
    print(f"Need ≥4 valid results (have {len(valid)})")

In [ ]:
#@title 📋 **LaTeX Table**
if summary:
    latex = f"""\\begin{{table}}[h]
\\centering
\\caption{{DiffuSVG Results (N={summary['N']})}}
\\begin{{tabular}}{{lc}}
\\toprule
Metric & Value \\\\
\\midrule
CLIP Score & {summary['CLIP Mean']:.2f} $\\pm$ {summary['CLIP Std']:.2f} \\\\
DINO Score & {summary['DINO Mean']:.3f} $\\pm$ {summary['DINO Std']:.3f} \\\\
Validity & {summary['Validity %']:.1f}\\% \\\\
Elements & {summary['Avg Elements']:.1f} \\\\
Time & {summary['Avg Time']:.1f}s \\\\
\\bottomrule
\\end{{tabular}}
\\end{{table}}"""
    print(latex)
    open('results.tex','w').write(latex)

In [ ]:
#@title 🔧 **LoRA Repair Pass** — upload adapter zip first, then run
# ─────────────────────────────────────────────────────────────────────────
# Fixes results where success=False or CLIP/DINO below threshold using the
# fine-tuned Qwen2-VL-2B LoRA adapter (text → SVG, no image needed).
# The adapter was retrained on QuiverAI geometric SVGs (short, clean paths)
# so path bounds are tighter: 1–20 paths, ≤3000 chars.
# Runs in-place on `results` so the Save cell below picks up the fixes.
# ─────────────────────────────────────────────────────────────────────────

import gc, re, zipfile
from pathlib import Path
from peft import PeftModel
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, BitsAndBytesConfig

# ── Thresholds ────────────────────────────────────────────────────────────
REPAIR_CLIP_THRESH  = 24.0
REPAIR_DINO_THRESH  = 0.35
LORA_ADAPTER_PATH   = "./qwen2vl_svg_lora/final_adapter"
LORA_BASE_MODEL     = "Qwen/Qwen2-VL-2B-Instruct"
MAX_PATHS_QUIVER    = 20    # QuiverAI SVGs have 1–15 paths typically
MAX_SVG_CHARS       = 3000  # QuiverAI SVGs are short (~200–800 chars)

# ── Step 0: Upload & unzip adapter (skip if already present) ─────────────
if not Path(LORA_ADAPTER_PATH).exists():
    print("Adapter not found — uploading zip...")
    from google.colab import files as _gfiles
    _up = _gfiles.upload()                        # select qwen2vl_svg_lora.zip
    for _fname in _up:
        if _fname.endswith(".zip"):
            with zipfile.ZipFile(_fname) as _z:
                _z.extractall(".")
            print(f"Extracted {_fname}")
            break
    if not Path(LORA_ADAPTER_PATH).exists():
        raise FileNotFoundError(f"Expected adapter at {LORA_ADAPTER_PATH} after extraction.")
else:
    print(f"Adapter found: {LORA_ADAPTER_PATH}")

# ── Step 1: Offload main Qwen2-VL-7B to free VRAM ────────────────────────
print("Offloading Qwen2-VL-7B to CPU...")
qwen_model.to("cpu")
gc.collect(); torch.cuda.empty_cache()
_free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1e9
print(f"VRAM free: {_free:.1f} GB")

# ── Step 2: Load fine-tuned Qwen2-VL-2B + LoRA ───────────────────────────
print(f"Loading {LORA_BASE_MODEL} + LoRA adapter...")
_quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
_base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    LORA_BASE_MODEL, quantization_config=_quant, device_map="auto",
)
ft_model = PeftModel.from_pretrained(_base_model, LORA_ADAPTER_PATH)
ft_model.eval()
ft_tok = AutoTokenizer.from_pretrained(LORA_ADAPTER_PATH)
print("Fine-tuned model ready.")

# ── Step 3: Text → SVG generation via LoRA ───────────────────────────────
_REPAIR_SYSTEM = (
    "You are an expert SVG code generator. "
    "Given a text description, output ONLY valid SVG code. "
    "Rules:\n"
    '- Start with: <svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg">\n'
    '- Use simple shapes: <rect>, <circle>, <ellipse>, <polygon>, <path>\n'
    '- Use solid hex fill colours (e.g., fill="#FF0000")\n'
    "- Keep it minimal: 5-20 elements\n"
    "- End with: </svg>\n"
    "Output the SVG code directly, no explanation."
)

def _lora_gen_svg(prompt: str) -> str:
    chat = ft_tok.apply_chat_template(
        [
            {"role": "system", "content": _REPAIR_SYSTEM},
            {"role": "user",   "content": f"Generate SVG for: {prompt}"},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    ids = ft_tok(chat, return_tensors="pt").to(ft_model.device)
    stop_id = ft_tok.encode("</svg>", add_special_tokens=False)[-1]
    with torch.inference_mode():
        out = ft_model.generate(
            **ids,
            max_new_tokens=1024,
            do_sample=False,
            repetition_penalty=1.5,
            eos_token_id=stop_id,
        )
    raw = ft_tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    # Strip markdown fences if present
    if "```" in raw:
        for part in raw.split("```"):
            p = part.strip().lstrip("svg").strip()
            if p.startswith("<svg"):
                raw = p; break
    m = re.search(r"<svg[\s>]", raw)
    if m:
        raw = raw[m.start():]
    if not raw.rstrip().endswith("</svg>"):
        raw = raw.rstrip() + "\n</svg>"
    return raw

# ── Step 4: Repair pass ───────────────────────────────────────────────────
_repaired = _kept_original = _out_of_range = 0

for r in tqdm(results, desc="Repair pass"):
    _is_bad = (
        not r.get("success")
        or r.get("clip", 99.0) < REPAIR_CLIP_THRESH
        or r.get("dino", 1.0)  < REPAIR_DINO_THRESH
    )
    if not _is_bad:
        continue

    _svg = _lora_gen_svg(r["prompt"])
    _n_paths = len(re.findall(r"<path", _svg))

    if not (1 <= _n_paths <= MAX_PATHS_QUIVER) or len(_svg) > MAX_SVG_CHARS:
        print(f"  ✗ {r['prompt'][:45]}  (paths={_n_paths}, chars={len(_svg)}, keeping original)")
        _out_of_range += 1
        continue

    # Re-score with CLIP + DINO
    _rendered = SVGUtils.render(_svg)
    if _rendered is None:
        print(f"  ✗ {r['prompt'][:45]}  (render failed)")
        _out_of_range += 1
        continue

    _clip = metrics.clip_score(_rendered, r["prompt"])
    _dino = metrics.dino_score(r["_img"], _rendered) if r.get("_img") else 0.0

    # Only accept if LoRA is actually better (or original had failed entirely)
    _orig_clip = r.get("clip", 0.0)
    _orig_dino = r.get("dino", 0.0)
    if r.get("success") and _clip < _orig_clip and _dino < _orig_dino:
        print(f"  ~ {r['prompt'][:45]}  LoRA worse (CLIP {_clip:.1f}<{_orig_clip:.1f}), keeping original")
        _kept_original += 1
        continue

    r["svg"]           = _svg
    r["success"]       = True
    r["clip"]          = _clip
    r["dino"]          = _dino
    r["elements"]      = SVGUtils.count_elements(_svg)
    r["lora_repaired"] = True
    r["_rendered"]     = _rendered
    _repaired += 1
    print(f"  ✓ {r['prompt'][:45]}  CLIP={_clip:.1f}  DINO={_dino:.3f}  paths={_n_paths}")

print(f"\nRepaired: {_repaired}  |  Kept original: {_kept_original}  |  Unrepairable: {_out_of_range}")

# ── Step 5: Recompute summary ─────────────────────────────────────────────
_valid_now = [r for r in results if r["success"]]
if _valid_now:
    summary.update({
        "Valid":         len(_valid_now),
        "Validity %":   100 * len(_valid_now) / len(results),
        "CLIP Mean":    np.mean([r["clip"]     for r in _valid_now]),
        "CLIP Std":     np.std( [r["clip"]     for r in _valid_now]),
        "DINO Mean":    np.mean([r["dino"]     for r in _valid_now]),
        "DINO Std":     np.std( [r["dino"]     for r in _valid_now]),
        "Avg Elements": np.mean([r["elements"] for r in _valid_now]),
        "LoRA Repaired": _repaired,
    })
    print(f"\nUpdated summary — Valid: {summary['Valid']}/{summary['N']}  "
          f"CLIP: {summary['CLIP Mean']:.2f}  DINO: {summary['DINO Mean']:.3f}")

# ── Step 6: Cleanup — restore main model ─────────────────────────────────
del ft_model, ft_tok, _base_model
gc.collect(); torch.cuda.empty_cache()
qwen_model.to("cuda")
print("Main model restored to GPU. Run Save cell next.")

In [ ]:
#@title 💾 **Save & Download**
# Prepare data (exclude PIL images)
export = [{'prompt': r['prompt'], 'success': r['success'], 'clip': r['clip'], 
           'dino': r['dino'], 'elements': r['elements'], 'time': r['time'],
           'svg': r['svg'], 'error': r.get('error')} for r in results]

with open('results.json', 'w') as f:
    json.dump({'summary': summary, 'results': export}, f, indent=2)
pd.DataFrame(export).to_csv('results.csv', index=False)

# Zip
import subprocess
files = ['results.json', 'results.csv'] + [f for f in ['plots.png','gallery.png','results.tex'] if os.path.exists(f)]
subprocess.run(['zip', '-q', 'results.zip'] + files)

from google.colab import files as gfiles
gfiles.download('results.zip')
print(f"✅ Downloading: {files}")